# EnsembleTreeNAM: Jointly trained TreeNAM ensemble

EnsembleTreeNAM trains several complete TreeNAM learners jointly and averages their predictions and term contributions. It is not bootstrap bagging or boosting.


## Model


For $M$ jointly optimized learners,

$$
\eta(x)=\frac{1}{M}\sum_{m=1}^{M}\eta_m(x),
\qquad
f_j(x_j)=\frac{1}{M}\sum_{m=1}^{M}f_{mj}(x_j).
$$


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import (
    EnsembleTreeNAMClassifier, EnsembleTreeNAMLSS, EnsembleTreeNAMRegressor,
)


model = EnsembleTreeNAMRegressor(
    num_estimators=3,
    aggregation="mean",
    tree_depth=3,
    tree_lamda=1e-3,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`num_estimators` controls jointly trained learners. Use `NeuralEnsemble` instead when independently initialized or bootstrapped fitted models are required.


In [ ]:
if RUN_TRAINING:
    display({"learners": model.get_params(deep=False)["num_estimators"]})
    components = model.predict_components(X_test)
    components.validate_additive_reconstruction()


## Task variants and limits

The joint ensemble is available for regression, classification, and LSS.
